# Project 1
### Shawn Ganz

I decided to use a custom dataset generated by Ai. This is the prompt that I used:

> I have a toy recommender system dataset with 10 users. I want to create a realistic user-item matrix with 10 movies total.
> 
> Please do the following:

> 1. Replace the 5 high-quality movies with 5 new well-regarded movies. For each movie, search for its Metacritic score and use that as a guide for how highly users should rate it.
> 2. Keep the same structure:
>    - 2 mid-tier movies (Metascore around 45-55)
>    - 3 low-quality/"garbage" movies (Metascore below 40)
> 3. Generate plausible 1-5 star ratings for all 10 users across all 10 movies.
> 4. Include some missing values (NaNs) to keep it realistic.
> 5. Use short snake_case column names for the movies (e.g. Dune_Part_Two, Everything_Everywhere_All_at_Once).
> 6. Use numeric user_ids from 1 to 10.
> 7. Make sure high-quality movies generally get high ratings (mostly 4s and 5s), mid movies get average ratings (~3), and garbage movies get low ratings (mostly 1s and 2s), with some natural variation between users.

## Code

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.precision', 2)

data = {
    "user_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],

    # === 5 High-Quality movies ===
    "Dune_Part_Two":                    [5, 4, np.nan, 5, 5, 4, 5, 4, 4, 5],
    "Everything_Everywhere_All_at_Once":[5, 5, 4, np.nan, 5, 3, 5, np.nan, 4, 5],
    "Top_Gun_Maverick":                 [4, 5, 4, 5, np.nan, 4, 5, 4, np.nan, 5],
    "Oppenheimer":                      [5, 4, np.nan, 5, 4, 3, 5, np.nan, 5, 5],
    "The_Batman":                       [4, 4, 3, 5, 4, np.nan, 4, 3, 4, np.nan],

    # === 2 Mid movies (Metascore ~49) ===
    "The_Gray_Man":  [4, 3, 3, 4, np.nan, 2, 4, 3, 3, 4],
    "Bullet_Train":  [3, 4, 2, np.nan, 3, 3, 4, 2, np.nan, 3],

    # === 3 Garbage movies (Metascore 26-35) ===
    "Morbius":       [2, 1, np.nan, 3, 2, 1, 2, np.nan, 1, 2],
    "Madame_Web":    [1, 2, 1, np.nan, 2, 1, np.nan, 2, 1, 1],
    "Cats":          [np.nan, 1, 2, 2, 1, np.nan, 1, 1, 2, 1]
}

df = pd.DataFrame(data)
df = df.set_index("user_id")

print(df)

         Dune_Part_Two  Everything_Everywhere_All_at_Once  Top_Gun_Maverick  Oppenheimer  The_Batman  The_Gray_Man  Bullet_Train  Morbius  Madame_Web  Cats
user_id                                                                                                                                                    
1                  5.0                                5.0               4.0          5.0         4.0           4.0           3.0      2.0         1.0   NaN
2                  4.0                                5.0               5.0          4.0         4.0           3.0           4.0      1.0         2.0   1.0
3                  NaN                                4.0               4.0          NaN         3.0           3.0           2.0      NaN         1.0   2.0
4                  5.0                                NaN               5.0          5.0         5.0           4.0           NaN      3.0         NaN   2.0
5                  5.0                                5.0       

### Calculations

In [2]:
# Create full user-item matrix
ratings_matrix = df.copy()

# Convert matrix to long format
ratings_long = (
    ratings_matrix
    .reset_index()
    .melt(
        id_vars="user_id",
        var_name="item_id",
        value_name="rating"
    )
    .dropna()
)

# Split known ratings into train/test
train_df, test_df = train_test_split(
    ratings_long,
    test_size=0.20,
    random_state=50
    
)

print(f'Shape | Training Data: {train_df.shape}')
print(f'Shape | Training Data: {test_df.shape}')
print('\n---\n')

# Raw Average
raw_average = train_df['rating'].mean()
print(f'Raw Average: {raw_average}')

# RMSE 

train_rmse_raw_average = np.sqrt(np.mean((train_df['rating'] - raw_average)**2))
test_rmse_raw_average = np.sqrt(np.mean((test_df['rating'] - raw_average)**2))

print(f'Raw Average | Train RMSE: {train_rmse_raw_average}')
print(f'Raw Average | Test RMSE: {test_rmse_raw_average}')

# Bias Value

user_bias = train_df.groupby("user_id")["rating"].mean() - raw_average
item_bias = train_df.groupby("item_id")["rating"].mean() - raw_average

print('\n---\n')
print(f'User Bias: {user_bias}')
print('\n---\n')
print(f'Item Bias: {item_bias}')
print('\n---\n')

# Baseline Predictor

train_df['baseline_predictor'] = raw_average + train_df['user_id'].map(user_bias) + train_df['item_id'].map(item_bias)
test_df['baseline_predictor'] = raw_average + test_df['user_id'].map(user_bias) + test_df['item_id'].map(item_bias)

train_df["baseline_predictor"] = train_df["baseline_predictor"].clip(lower=1, upper=5)
test_df["baseline_predictor"] = test_df["baseline_predictor"].clip(lower=1, upper=5)

print(f'Training df w/ Baseline Predictor: \n {train_df}')
print('\n---\n')
print(f'Testing df w/ Baseline Predictor: \n {test_df}')
print('\n---\n')

# Baseline RMSE

train_rmse_baseline = np.sqrt(np.mean((train_df['rating'] - train_df['baseline_predictor'])**2))
test_rmse_baseline = np.sqrt(np.mean((test_df['rating'] - test_df['baseline_predictor'])**2))

print(f'Baseline | Train RMSE: {train_rmse_baseline}')
print(f'Baseline | Test RMSE: {test_rmse_baseline}')
print(f'Raw Average | Train RMSE: {train_rmse_raw_average}')
print(f'Raw Average | Test RMSE: {test_rmse_raw_average}')
print('\n---\n')
print(f'Percent Improvement over Raw | Train: {(1-train_rmse_baseline/train_rmse_raw_average)*100}')
print(f'Percent Improvement over Raw | Test: {(1-test_rmse_baseline/test_rmse_raw_average)*100}')

Shape | Training Data: (65, 3)
Shape | Training Data: (17, 3)

---

Raw Average: 3.3076923076923075
Raw Average | Train RMSE: 1.3803352649943357
Raw Average | Test RMSE: 1.4782328279975314

---

User Bias: user_id
1     0.55
2     0.19
3    -0.59
4     0.69
5    -0.06
6    -0.59
7     0.55
8    -0.51
9    -0.31
10    0.29
Name: rating, dtype: float64

---

Item Bias: item_id
Bullet_Train                        -0.45
Cats                                -1.71
Dune_Part_Two                        1.19
Everything_Everywhere_All_at_Once    1.12
Madame_Web                          -2.02
Morbius                             -1.64
Oppenheimer                          1.19
The_Batman                           0.36
The_Gray_Man                         0.03
Top_Gun_Maverick                     1.29
Name: rating, dtype: float64

---

Training df w/ Baseline Predictor: 
     user_id                            item_id  rating  baseline_predictor
10        1  Everything_Everywhere_All_at_Once     5.0 

## Summarization

Using the baseline predictor reduced the RMSE by approximately 65.5% on the training set and 66.1% on the test set compared to using the raw average alone. This shows that accounting for user and item biases leads to significantly better rating predictions.

The item biases aligned well with the quality of the movies. High-quality films such as Dune: Part Two, Oppenheimer, and Top Gun: Maverick received strong positive biases, while poorly received movies like Madame Web, Cats, and Morbius received large negative biases. These results suggest that the baseline model successfully captured meaningful differences between users and items.

The summary statistics reveal noticeable differences in rating behavior across both users and items. In the training set, average ratings per user ranged from 2.71 to 4.00, while item averages showed a clear distinction between well-received movies (such as Top Gun: Maverick and Dune: Part Two, averaging above 4.5) and poorly received ones (such as Madame Web and Cats, averaging below 1.7). A similar pattern was observed in the test set, supporting the presence of meaningful user and item biases in the data.

In [14]:
print(f'Shape | Training Data: {train_df.shape}')
print(f'Shape | Training Data: {test_df.shape}')
print('\n---\n')

print(f'Raw Average: {raw_average}')

print(f'Raw Average | Train RMSE: {train_rmse_raw_average}')
print(f'Raw Average | Test RMSE: {test_rmse_raw_average}')

print('\n---\n')
print(f'User Bias: {user_bias}')
print('\n---\n')
print(f'Item Bias: {item_bias}')
print('\n---\n')

print(f'Training df w/ Baseline Predictor: \n {train_df}')
print('\n---\n')
print(f'Testing df w/ Baseline Predictor: \n {test_df}')
print('\n---\n')

print(f'Baseline | Train RMSE: {train_rmse_baseline}')
print(f'Baseline | Test RMSE: {test_rmse_baseline}')
print(f'Raw Average | Train RMSE: {train_rmse_raw_average}')
print(f'Raw Average | Test RMSE: {test_rmse_raw_average}')
print('\n---\n')

print(f'Percent Improvement over Raw | Train: {(1 - train_rmse_baseline / train_rmse_raw_average) * 100}')
print(f'Percent Improvement over Raw | Test: {(1 - test_rmse_baseline / test_rmse_raw_average) * 100}')
print('\n---\n')

print("Ratings per User (Train) - Sorted by Highest Average Rating:")
print(train_df.groupby("user_id")["rating"].agg(["count", "mean"])
      .sort_values("mean", ascending=False))
print('\n')

print("Ratings per Item (Train) - Sorted by Highest Average Rating:")
print(train_df.groupby("item_id")["rating"].agg(["count", "mean"])
      .sort_values("mean", ascending=False))
print('\n---\n')

print("Ratings per User (Test) - Sorted by Highest Average Rating:")
print(test_df.groupby("user_id")["rating"].agg(["count", "mean"])
      .sort_values("mean", ascending=False))
print('\n')

print("Ratings per Item (Test) - Sorted by Highest Average Rating:")
print(test_df.groupby("item_id")["rating"].agg(["count", "mean"])
      .sort_values("mean", ascending=False))


Shape | Training Data: (65, 4)
Shape | Training Data: (17, 4)

---

Raw Average: 3.3076923076923075
Raw Average | Train RMSE: 1.3803352649943357
Raw Average | Test RMSE: 1.4782328279975314

---

User Bias: user_id
1     0.55
2     0.19
3    -0.59
4     0.69
5    -0.06
6    -0.59
7     0.55
8    -0.51
9    -0.31
10    0.29
Name: rating, dtype: float64

---

Item Bias: item_id
Bullet_Train                        -0.45
Cats                                -1.71
Dune_Part_Two                        1.19
Everything_Everywhere_All_at_Once    1.12
Madame_Web                          -2.02
Morbius                             -1.64
Oppenheimer                          1.19
The_Batman                           0.36
The_Gray_Man                         0.03
Top_Gun_Maverick                     1.29
Name: rating, dtype: float64

---

Training df w/ Baseline Predictor: 
     user_id                            item_id  rating  baseline_predictor
10        1  Everything_Everywhere_All_at_Once     5.0 

## Extras

Including some tasks that were not requested, just for practice. Also, including a point of failure I had earlier in creating this recommender.

### Final Dataframe

I decided to fill in the missing values (NAs) in the original dataset in the code block below. After confirming in the previous section that the baseline model showed strong improvement over the raw average on both the training and test sets, the same approach was applied to the full dataset. User and item biases were calculated using all known ratings, a prediction matrix was created, and the missing values in the original matrix were then replaced with the predicted values.

In [3]:
print('Initial Dataframe: \n', df, '\n\n---\n')

column_order = [
    "Dune_Part_Two",
    "Everything_Everywhere_All_at_Once",
    "Top_Gun_Maverick",
    "Oppenheimer",
    "The_Batman",
    "The_Gray_Man",
    "Bullet_Train",
    "Morbius",
    "Madame_Web",
    "Cats"
]

final_long = (
    df.reset_index()
    .melt(
        id_vars="user_id",
        var_name="item_id",
        value_name="rating"
    )
)

full_raw_average = df.stack().mean()

final_long['baseline_predictor'] = (raw_average + final_long['user_id'].map(user_bias).fillna(0) + final_long['item_id'].map(item_bias).fillna(0))

final_long['baseline_predictor'] = final_long['baseline_predictor'].clip(1, 5)

predicted_matrix = (
    final_long.pivot(
        index='user_id', 
        columns='item_id', 
        values='baseline_predictor'
    )
    .round(2)
    .sort_index(axis=0)
    [column_order]
)

print('Prediction Dataframe: \n', predicted_matrix, '\n\n---\n')

final_matrix = df.fillna(predicted_matrix)

final_matrix = (
    df.fillna(predicted_matrix)
    .sort_index(axis=0)
    [column_order]
)

print('Final Dataframe:\n',final_matrix)



Initial Dataframe: 
          Dune_Part_Two  Everything_Everywhere_All_at_Once  Top_Gun_Maverick  Oppenheimer  The_Batman  The_Gray_Man  Bullet_Train  Morbius  Madame_Web  Cats
user_id                                                                                                                                                    
1                  5.0                                5.0               4.0          5.0         4.0           4.0           3.0      2.0         1.0   NaN
2                  4.0                                5.0               5.0          4.0         4.0           3.0           4.0      1.0         2.0   1.0
3                  NaN                                4.0               4.0          NaN         3.0           3.0           2.0      NaN         1.0   2.0
4                  5.0                                NaN               5.0          5.0         5.0           4.0           NaN      3.0         NaN   2.0
5                  5.0                     

### Previous Issues

I tried to get creative by randomizing a dataset where the user values can be adjusted via the `n_users` variable. However, I noticed that the Baseline RMSE for the test was always negative. After a long while, I then realized that randomized data with no coherent pattern will cause issues. The reason is because the bias for both the training dataset's `user` and the `item` (movie ratings) are just noise, so when we fit the test with these biases, the baseline rmse will always perform worse than just the raw average when we do the percentage improvement calculation. 

In [ ]:
n_users = 100

users = [f"U{i}" for i in range(1, n_users + 1)]
movies = ["Inception", "The Matrix", "Interstellar", "Dune", "Oppenheimer"]

# Random ratings from 1 to 5
ratings = np.random.randint(1, 6, size=(len(users), len(movies))).astype(float)

# Every even row gets a n/a
for i in range(1, len(users), 2):
    n_missing = np.random.choice([1, 2])

    missing_cols = np.random.choice(
        len(movies),
        size=n_missing,
        replace=False
    )

    ratings[i, missing_cols] = np.nan

df = pd.DataFrame(ratings, columns=movies)
df.insert(0, "user_id", users)

df

,user_id,Inception,The Matrix,Interstellar,Dune,Oppenheimer
0,U1,5.0,5.0,5.0,3.0,3.0
1,U2,3.0,NaN,2.0,3.0,NaN
2,U3,4.0,5.0,3.0,1.0,3.0
3,U4,NaN,5.0,2.0,3.0,1.0
4,U5,1.0,3.0,2.0,5.0,1.0
...,...,...,...,...,...,...
495,U496,NaN,1.0,4.0,4.0,NaN
496,U497,2.0,5.0,4.0,1.0,4.0
497,U498,3.0,4.0,NaN,NaN,2.0
498,U499,4.0,5.0,3.0,5.0,3.0
